
# Agentic AI - Evaluation

## 1. Introduction 

This notebook implements the component-level evaluation within an agentic workflow that uses a structured financial research agent. The research agent performs the following tasks:

1. Search the web for informatio on a research topic
2. Write a structured draft based on the information
3. Reflect and improve
4. Publish the final response to the topic

The agent produces structured output:

{
  "final_answer": "... (≤ 250 words)",
  "key_facts": ["...", "...", "..."],
  "sources": ["url1", "url2"]
}

At each stage of the process, we will design **component-level evaluation** to check the quality of the specific task the agent performs. Systematic evaluation is demonstrated along two core dimensions:

1. **Per-example ground truth vs no ground truth**
2. **Code-based evaluation vs LLM-as-judge evaluation**

Evals are applied to two sets of queries. They are modified dep

- One offline golden example (with ground truth)
- One live user query (no ground truth)


## 2. Evaluation Framework

We organize evaluation into four quadrants:

|                         | Code-Based (Deterministic) | LLM-as-Judge |
|-------------------------|----------------------------|--------------|
| **Ground Truth Exists** | Fact recall                | Semantic fact coverage |
| **No Ground Truth**     | Structural checks          | Rubric grading |

Each quadrant serves a different purpose.

### 2.1 Component-Level Evaluation

The agent has three core components:

1. Structured `key_facts`
2. Narrative `final_answer`
3. Reflection mechanism

Each of these are evaluated differently.

#### 2.1.1 Structured Key Facts

##### When ground truth exists (offline benchmark)

We evaluate `key_facts` using:

• Deterministic fact recall  
• Optional LLM-based semantic coverage  

Recall = matched_expected_facts / total_expected_facts

An optional LLM judge checks semantic coverage when wording differs but meaning aligns — ensuring deterministic matching is not overly strict.

#### 2.1.2 Narrative Final Answer

The `final_answer` is open-ended and explanatory, so we avoid deterministic paragraph matching.

Instead:

• With ground truth → use rubric scoring  
• Without ground truth → rely on structural checks + rubric scoring  

##### Code-Based Structural Checks

We validate:

• Word limit (≤ 250 words)  
• Minimum key facts  
• At least one source  
• Valid JSON format  

These ensure output reliability, not correctness.

##### LLM-as-Judge Rubric Evaluation

Used when semantic reasoning must be assessed.We score:

• Accuracy  
• Completeness  
• Clarity  
• Hallucination risk  

This approximates expert review when deterministic comparison is not possible.

#### 2.1.3 Reflection Mechanism

Reflection is evaluated by comparing Draft vs Final. It is measured using:

• Change in deterministic recall  
• Change in rubric scores  
• Structural compliance improvements  

This makes reasoning improvement measurable rather than assumed.

##### Judge Reliability

LLM-as-judge is complementary.

We monitor:

• Consistency across runs  
• Agreement with deterministic metrics  
• Sensitivity to prompt design  

Judge scores inform evaluation — they do not replace measurable metrics.

## 3. Setup: Initialize Environment and Clients

In this step, we import the key libraries that support the agent workflow:

- **`openai`**: Used to interact with the OpenAI API for generating responses.
- **`tavily`**: Used to perform web search and retrieve relevant source snippets for the research question.
- **`json`**: Provides functions to read and write JSON data, which is useful for handling the structured outputs returned by the LLM.
- **`utils`**: A custom helper module for this project. It includes utility functions for dataset handling, result formatting, and lightweight evaluation support.

The OpenAI and Tavily API keys must be provided in a `.env` file.

We also use a golden dataset of 10 benchmark examples stored in:

`golden_dataset.json`

Each example contains:
- A research question  
- A canonical list of expected key facts  

This dataset enables controlled, reproducible offline evaluation.


In [1]:
# Standard library imports
import sys
sys.path.append("../../agentic-ai")

import os
import json
import re
import string

from dotenv import load_dotenv
from openai import OpenAI
from tavily import TavilyClient

# User-Defined Functions
from utils import utils
from research_agent.search import search_web
from research_agent.write import write_structured_answer
from research_agent.reflect import reflect_and_improve
from research_agent.llm_wrapper import LLMWrapper

In [2]:
# Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

# Initialize clients
llm = LLMWrapper(api_key=OPENAI_API_KEY)

tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

In [3]:
# Check the golden dataset
with open("../data/golden_dataset.json") as f:
    golden_data = json.load(f)
golden_data

[{'question': 'What were the main causes of the 2008 global financial crisis?',
  'expected_key_facts': ['subprime mortgage lending',
   'housing bubble',
   'excessive leverage in financial institutions',
   'securitization of mortgage-backed securities',
   'collapse of Lehman Brothers']},
 {'question': 'What caused the dot-com bubble to burst in 2000?',
  'expected_key_facts': ['overvaluation of technology stocks',
   'speculative investment',
   'lack of profitable business models',
   'tightening of monetary policy',
   'market correction']},
 {'question': 'What led to the 1997 Asian financial crisis?',
  'expected_key_facts': ['excessive foreign borrowing',
   'fixed exchange rate regimes',
   'weak financial regulation',
   'speculative capital flows',
   'sudden capital flight']},
 {'question': 'What were the main factors behind the European sovereign debt crisis?',
  'expected_key_facts': ['high government debt levels',
   'fiscal deficits',
   'weak economic growth',
   'bank

## 4. Evaluation 

### 4.1 Offline Example (With Ground Truth)

In [4]:
item = golden_data[0]

question = item["question"]
expected_key_facts = item["expected_key_facts"]

snippets = search_web(question, tavily_client)
draft = write_structured_answer(question, snippets, llm)
final = reflect_and_improve(question, snippets, draft, llm)

utils.print_html(final)

In [5]:
def normalize_text(text: str) -> str:
    """Normalize text by converting to lowercase, removing punctuation, and stripping whitespace.

    Args:
        text (str): The input text to normalize.

    Returns:
        str: The normalized text.
    """
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compute_fact_recall(generated_facts: list[str], expected_facts: list[str]) -> float:
    """
    Compute the recall of generated facts against expected facts.

    Args:
        generated_facts(list[str]): The list of facts generated by the agent.
        expected_facts(list[str]): The list of expected key facts from the golden dataset.

    Returns:
        The recall score, calculated as the number of matched facts divided by the total number of expected facts.
    """
    normalized_generated = [normalize_text(f) for f in generated_facts]
    matched = 0

    for expected in expected_facts:
        normalized_expected = normalize_text(expected)
        if any(normalized_expected in gen for gen in normalized_generated):
            matched += 1

    return matched / len(expected_facts)




In [6]:
draft_recall = compute_fact_recall(draft["key_facts"], expected_key_facts)
final_recall = compute_fact_recall(final["key_facts"], expected_key_facts)

print("\nOffline Metrics")
print("Draft Recall:", draft_recall)
print("Final Recall:", final_recall)
print("Improvement:", final_recall - draft_recall)


Offline Metrics
Draft Recall: 0.2
Final Recall: 0.2
Improvement: 0.0


### 4.2 Evaluation - Live User Query (No Ground Truth)

In [7]:

user_question = "Why did Fed raise interest rates in 2023?"

snippets = search_web(user_question, tavily_client)
draft_live = write_structured_answer(user_question, snippets, llm)
final_live = reflect_and_improve(user_question, snippets, draft_live, llm)

utils.print_html("Final Answer:", final_live)


In [8]:

def check_constraints(output: dict) -> dict:
    """Check if the output meets the defined constraints.

    Args:
        output (dict): The output dictionary containing 'key_facts', 'sources', and 'final_answer'.

    Returns:
        dict: A dictionary with boolean values indicating whether each constraint is met.
    """
    return {
        "min_key_facts": len(output["key_facts"]) >= 3,
        "min_sources": len(output["sources"]) >= 1,
        "word_limit_ok": len(output["final_answer"].split()) <= 250
    }


In [9]:
print("\nLive Structural Checks")
print(check_constraints(final_live))


Live Structural Checks
{'min_key_facts': True, 'min_sources': True, 'word_limit_ok': True}


### 5. Final Takeaways